# Sleep Planner

In [ ]:
import math
from datetime import datetime, timedelta

In [ ]:
COFFEES = [
    ("กาแฟดำชงผ่านกระดาษ (Drip)", 240, 95, 96),
    ("กาแฟดำ Drip แก้วใหญ่", 350, 140, 140),
    ("Espresso 1 Shot", 30, 63, 63),
    ("Espresso 2 Shots", 60, 126, 126),
    ("Americano 1 Shot", 240, 63, 63),
    ("Americano 2 Shots", 350, 126, 126),
    ("Latte 1 Shot", 240, 63, 63),
    ("Cappuccino 1 Shot", 180, 63, 63),
    ("Cold Brew", 240, 100, 200),
    ("Instant Coffee", 240, 62, 62),
    ("Decaf Coffee", 240, 1, 5),
]

TARGETS = [
    (100, "เหลือไม่เกิน 100 mg"),
    (50, "เหลือไม่เกิน 50 mg"),
    (1, "ใกล้หมด (ต่ำกว่า 1 mg)")
]

In [ ]:
def convert_time(value):
    value = value.strip()

    # รองรับ 12:00, 1200, 12000, 830
    if ":" in value:
        parts = value.split(":")
        if len(parts) != 2:
            raise ValueError("รูปแบบเวลาไม่ถูกต้อง")
        hour, minute = parts
    else:
        if len(value) == 5 and value.endswith("0"):
            value = value[:-1]

        if not value.isdigit():
            raise ValueError("กรุณากรอกเวลาเป็นตัวเลข")

        if len(value) <= 2:
            hour, minute = value, "00"
        elif len(value) in (3, 4):
            hour, minute = value[:-2], value[-2:]
        else:
            raise ValueError("รูปแบบเวลาไม่ถูกต้อง")

    hour = int(hour)
    minute = int(minute)

    if not (0 <= hour <= 23 and 0 <= minute <= 59):
        raise ValueError("ชั่วโมงหรือนาทีไม่ถูกต้อง")

    return datetime.strptime(f"{hour:02d}:{minute:02d}", "%H:%M")

In [ ]:
def calc_vd(weight):

    return weight * 0.6


def calc_c0(dose, vd):
    """ความเข้มข้นเริ่มต้น C0 (สมมติว่าคาเฟอีนเข้าสู่ร่างกายทันที)"""
    return dose / vd


def calc_cl(weight, age):

    cl_per_kg = 0.06 if age >= 65 else 0.085
    return cl_per_kg * weight


def calc_half_life(vd, cl):

    return math.log(2) * vd / cl


def calc_ke(t_half):

    return math.log(2) / t_half


def calc_pk(weight, age, dose):

    vd = calc_vd(weight)
    cl = calc_cl(weight, age)
    t_half = calc_half_life(vd, cl)
    return {
        "Vd": vd,
        "C0": calc_c0(dose, vd),
        "CL": cl,
        "t_half": t_half,
        "ke": calc_ke(t_half),
    }


def minutes_to_target(dose, ke, target):

    hours = math.log(dose / target) / ke
    if target == 1:
        # ปัดไปนาทีถัดไป เพื่อให้ต่ำกว่า 1 mg จริง
        return max(0, math.floor(hours * 60) + 1)
    return max(0, math.ceil(hours * 60))


def calc_alerts(start_time, dose, ke, targets=TARGETS):
    
    alerts = []
    for target, label in targets:
        minutes = minutes_to_target(dose, ke, target)
        end_time = start_time + timedelta(minutes=minutes)
        alerts.append({
            "label": label,
            "minutes": minutes,
            "end_time": end_time,
            "days_later": (end_time.date() - start_time.date()).days,
        })
    return alerts

In [ ]:
def show_coffee_menu(coffees=COFFEES):
    print("\nเลือกกาแฟ 1 แก้ว/รายการ (ปริมาณคาเฟอีนโดยประมาณ)")
    for number, (name, volume, low, high) in enumerate(coffees, 1):
        amount = str(low) if low == high else f"{low}–{high}"
        print(f"{number:2}. {name} | {volume} ml | {amount} mg")


def day_label(days_later):
    if days_later == 0:
        return "วันเดียวกัน"
    if days_later == 1:
        return "วันถัดไป"
    return f"อีก {days_later} วัน"


def show_result(weight, age, coffee_name, volume, dose, start_time, pk, alerts):
    print("\n========== ผลการคำนวณ ==========")
    print(f"น้ำหนัก: {weight:.2f} kg")
    print(f"อายุ: {age} ปี")
    print(f"กาแฟ: {coffee_name} ({volume} ml)")
    print(f"คาเฟอีนเริ่มต้นโดยประมาณ: {dose:.2f} mg")
    print(f"เวลาที่ดื่ม: {start_time:%H:%M} น.")

    print("----------------------------------")
    print(f"Vd = {pk['Vd']:.2f} L")
    print(f"C0 = {pk['C0']:.2f} mg/L (ความเข้มข้นเริ่มต้นโดยประมาณ)")
    print(f"CL = {pk['CL']:.3f} L/h")
    print(f"Half-life = {pk['t_half']:.2f} ชั่วโมง")
    print(f"ke = {pk['ke']:.4f} 1/h")
    print("----------------------------------")

    print("เวลาแจ้งเตือนโดยประมาณ (แสดงผลใน Notebook)")
    for a in alerts:
        m = a["minutes"]
        if m == 0:
            print(f"• {a['label']}: ตั้งแต่เวลาดื่ม {start_time:%H:%M} น. (วันเดียวกัน)")
        else:
            print(f"• {a['label']}: {a['end_time']:%H:%M} น. ({day_label(a['days_later'])}) "
                  f"หลังดื่ม {m // 60} ชั่วโมง {m % 60} นาที")

    print("คาเฟอีนไม่ลดถึง 0 mg พอดีตามสมการนี้; ใกล้หมดหมายถึงต่ำกว่า 1 mg")
    print("เป็นค่าประมาณจากแบบจำลองเดิม ไม่ใช่เวลาที่รับรองว่าปลอดคาเฟอีน")
    print("ผลจริงขึ้นอยู่กับแต่ละบุคคลและปริมาณคาเฟอีนจริงในกาแฟ")
    print("==================================")

In [ ]:
def get_inputs():
    """ถามผู้ใช้จนกว่าจะกรอกถูก คืนค่า (weight, age, coffee_name, volume, dose, start_time)"""
    while True:
        try:
            weight = float(input("น้ำหนักตัว (kg): "))
            age = int(input("อายุ (ปี): "))

            show_coffee_menu()
            choice = int(input(f"เลือกหมายเลขกาแฟ (1–{len(COFFEES)}): "))
            if not 1 <= choice <= len(COFFEES):
                raise ValueError(f"กรุณาเลือกหมายเลขกาแฟ 1–{len(COFFEES)}")
            coffee_name, volume, low, high = COFFEES[choice - 1]

            dose = high
            if low != high:
                print(f"รายการนี้มีคาเฟอีนประมาณ {low}–{high} mg")
                estimate = input("ใช้ค่า 1 = ต่ำสุด, 2 = สูงสุด (Enter = สูงสุด): ").strip()
                if estimate not in ("", "1", "2"):
                    raise ValueError("กรุณาเลือก 1 หรือ 2")
                dose = low if estimate == "1" else high

            drink_time = input("เวลาที่ดื่ม (เช่น 12:00): ")

            if not math.isfinite(weight) or weight <= 0 or age < 0:
                raise ValueError("ข้อมูลไม่ถูกต้อง")

            start_time = convert_time(drink_time)
            return weight, age, coffee_name, volume, dose, start_time

        except ValueError as error:
            print("ข้อมูลไม่ถูกต้อง:", error)
            print("กรุณากรอกข้อมูลใหม่อีกครั้ง\n")

In [ ]:
def main():
    weight, age, coffee_name, volume, dose, start_time = get_inputs()  # 1. รับค่า
    pk = calc_pk(weight, age, dose)                                    # 2. คำนวณ
    alerts = calc_alerts(start_time, dose, pk["ke"])
    show_result(weight, age, coffee_name, volume, dose, start_time, pk, alerts)  # 3. แสดงผล


main()